# Voice cloning (Chatterbox) — Colab entrypoint

Runs the same `gpu/voice_chatterbox.py` used locally, as an importable function
rather than a subprocess (Colab has one environment, so there's no isolated
stage venv to shell out to — see `core/envs.py`).

Colab is ephemeral: this notebook mounts Drive first and writes every output
there, so a disconnect never loses finished work.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

In [ ]:
REPO_URL = "<your fork/clone URL>"  # e.g. https://github.com/you/presenter-video.git
REPO_DIR = "/content/presenter-video"

import os
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/presenter-video'
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
os.environ['PRESENTER_DRIVE_DIR'] = DRIVE_DIR
os.environ['HF_HOME'] = f'{DRIVE_DIR}/weights/hf'   # cache HF downloads on Drive across sessions
os.environ['TORCH_HOME'] = f'{DRIVE_DIR}/weights/torch'

In [ ]:
# First cell rule from CLAUDE.md: always install from requirements before anything else.
!pip install -q -r requirements.txt
!pip install -q -r requirements/voice.txt

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

from gpu.voice_chatterbox import run

text = "Hello! This is a self test of the presenter video pipeline running on Colab."
reference_audio = f"{REPO_DIR}/assets/samples/placeholder_voice.wav"  # swap for a real, consented sample
out_path = f"{DRIVE_DIR}/voice_test.wav"

run(text, reference_audio, out_path, device="cuda")
print("wrote", out_path)

In [ ]:
from IPython.display import Audio
Audio(out_path)